In [ ]:
from pathlib import Path
import itertools
import subprocess
import sys
from datetime import datetime

project_dir = Path("/scratch/rhm4nj/gpu_arch/gsplat-fork")
# project_dir = Path("/bigtemp/rhm4nj/gpu_arch/project/gsplat-fork")
TEMPLATE_PATH = str(project_dir/ "scripts/slurm/profile_gsplat_template_rivanna.slurm")
mode = "zip"   # grid or zip - grid does cartesian product of all params, zip pairs them by index (so all lists must be same length)

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
base_dir = "outputs" / Path(timestamp)
slurm_dir = base_dir / "slurm_scripts"
slurm_out_dir = base_dir / "slurm_out"
slurm_err_dir = base_dir / "slurm_err"
config_dir = base_dir / "configs"
results_dir = project_dir / "scripts/slurm" / base_dir / "results"

slurm_dir.mkdir(parents=True, exist_ok=False)
slurm_out_dir.mkdir(parents=True, exist_ok=False)
slurm_err_dir.mkdir(parents=True, exist_ok=False)
config_dir.mkdir(parents=True, exist_ok=False)
results_dir.mkdir(parents=True, exist_ok=False)

fixed_params = {
    "script_path": project_dir / "examples/benchmarks/custom_mod.sh",
    "partition": "gpu",
    "log_out": slurm_out_dir / "%x-%j.out",
    "log_err": slurm_err_dir / "%x-%j.err",
    "module_loads": "module load cuda\nmodule load nsight-systems",
    "trace": "cuda,osrt,nvtx",
    "sample": "cpu",
    # "load_env_cmd": "source /bigtemp/rhm4nj/gpu_arch/project/gsplat-env/bin/activate", # swap out with conda if ur using it
    "project_dir": project_dir,
    "gpus": 1,
    "cpus": 8,
    "mem": "32G",
    "time": "02:00:00",
    "force_overwrite": "true",
    "gpu_type": "a6000", # Must ALL runn the same GPU - you can add more GPU specifications, like ammount of GPU memory - https://www.cs.virginia.edu/computing/doku.php?id=compute_slurm
    "scenes": "garden",
    # profile_output is set per-job automatically from profile_prefix + sweep params
}

# --- Profile output naming ---
profile_prefix = "profile_gsplat"

def _abbrev(arg: str) -> str:
    """First letter of each word in a --kebab-case arg. e.g. --enable-frustum-culling -> efc"""
    return "".join(w[0] for w in arg.lstrip("-").split("-"))

def make_profile_output(prefix: str, keys: list, combo: tuple) -> str:
    """Build a name like: profile_gsplat_os-1_efc-True_eic-False"""
    parts = [f"{_abbrev(k)}-{v}" for k, v in zip(keys, combo)]
    return "_".join([prefix] + parts)

scenes = ["garden", "bicycle", "stump", "bonsai", "counter", "kitchen", "room"]

sweep_params = {
    "--optimizer-stride": [0, 1],
    "--enable-frustum-culling": [True, False],
    "--enable-input-cache": [True, False]
}

keys = list(sweep_params.keys())
values = list(sweep_params.values())

if mode == "grid": combinations = list(itertools.product(*values))

elif mode == "zip":
    lengths = [len(v) for v in values]
    if len(set(lengths)) != 1:
        raise ValueError("all sweep parameter lists must have same length in zip mode.")
    combinations = list(zip(*values))

else:
    raise ValueError("most has to b 'grid' or 'zip'")

total_jobs = len(combinations)
print(f"\ntotal jobs to generate: {total_jobs}")
print("\nExample profile_output names:")
for combo in combinations:
    print(" ", make_profile_output(profile_prefix, keys, combo))

In [ ]:
from pathlib import Path
import json

folders = [Path(slurm_dir), Path(config_dir)]
for folder in folders:
    if any(folder.iterdir()):
        raise RuntimeError("Output folder already has outputs - re-run above cell")

do_run = False
confirm = input("do you want to continue? (y/n): ") # "yes" runs scripts, "no" stll generates them but doesn't run
do_run = confirm.lower() == "y"

print("generating and submitting jobs...\n")

template = Path(TEMPLATE_PATH).read_text()
if "gpu_type" not in sweep_params and "gpu_type" not in fixed_params:
    template = template.replace(r'#SBATCH --constraint="{gpu_type}"', "")

job_map = {}

for idx, combo in enumerate(combinations):
    job_params = dict(zip(keys, combo))
    job_params["profile_output"] = make_profile_output(profile_prefix, keys, combo)
    job_params.update(fixed_params)
    filled = template.format(**job_params)
    job_file = slurm_dir / f"job_{idx}.slurm"
    job_file.write_text(filled)
    cfg_file = config_dir / f"job_{idx}.cfg"

    with open(cfg_file, "w") as f:
        for k, v in job_params.items():
            f.write(f"{k}={v}\n")

    job_map[f"job_{idx}"] = job_params["profile_output"]
    print(f"  job_{idx}: {job_params['profile_output']}")
    if do_run: 
        subprocess.run(["sbatch", str(job_file)])
        print("Ran", "sbatch", str(job_file))

# Write job_map.json to the run's base directory
job_map_path = base_dir / "job_map.json"
job_map_path.write_text(json.dumps(job_map, indent=2))
print(f"\nWrote job map → {job_map_path}")
print("\nAll jobs generated successfully.")